In [ ]:
from sklearn import datasets
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

X, y = datasets.load_iris(return_X_y=True)

X_train, X_test, y_train, y_test = train_test_split(
    X, 
    y, 
    test_size=0.2, 
    random_state=42
)



# Entrenar el modelo
params = {
    "random_state": 42,
    "max_iter": 1000
}
# Usamos **params para 
# desempaquetar el diccionario como argumentos
model = LogisticRegression(**params)
model.fit(X_train, y_train)
    
# Predecir en el set de prueba
y_pred = model.predict(X_test)

# Calcular métricas de evaluación
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average='weighted')
recall = recall_score(y_test, y_pred, average='weighted')
f1 = f1_score(y_test, y_pred, average='weighted')

print(f"✅ Accuracy: {accuracy:.4f}")
print(f"✅ Precision: {precision:.4f}")
print(f"✅ Recall: {recall:.4f}")
print(f"✅ F1: {f1:.4f}")

El automl.classification() solo configura un trabajo de AutoML, NO lo ejecuta. 

Y cuando lo envíes con mlclient.jobs.create_or_update(job), Azure ML lo ejecutará como un job independiente con su propio tracking.

Es como si el mlflow.start_run() de este job estubiera dentro del job de AutoML de mlclient.jobs.create_or_update(job).

In [ ]:
import mlflow
import pandas as pd
from mlflow.models import infer_signature

from azure.ai.ml import MLClient
from azure.ai.ml import automl
from azure.ai.ml import Input
from azure.ai.ml.entities import Data
from azure.ai.ml.constants import AssetTypes
from azure.identity import DefaultAzureCredential
import os

mlclient = MLClient(
    credential=DefaultAzureCredential(),
    subscription_id=os.getenv("AZURE_SUBSCRIPTION_ID"),
    resource_group_name=os.getenv("AZURE_RESOURCE_GROUP"),
    workspace_name=os.getenv("AZURE_WORKSPACE_NAME")
)
# NO usar mlflow.set_tracking_uri() cuando se usa azureml-mlflow
# El plugin azureml-mlflow ya configura la conexión automáticamente.
# Si se asigna manualmente, da conflicto con azureml_artifacts_builder.
# POR LO TANTO, NO usar mlflow.set_tracking_uri() cuando se usa azureml-mlflow

mlflow.set_experiment(experiment_name="iris-classifier_azureML")
#-----


# cargamos el dataset a AzureML usando la estructura de tabla
dataset = Data(
    name="iris-dataset",
    version="1",
    description="Dataset de iris para AutoML",
    path="./data/iris-data",
    type=AssetTypes.MLTABLE
)
#enviar
mlclient.data.create_or_update(dataset)
#-----

# Configurar el Input para AutoML ---------------------
my_training_data_input = Input(
    type=AssetTypes.MLTABLE,
    path="azureml:iris-dataset:1"
)

# Configurar la tarea de clasificación
job = automl.classification(
    experiment_name="iris-automl-classification",
    training_data=my_training_data_input,  # ← Usamos el Input, NO el objeto Data
    target_column_name="",
    compute="cpu-cluster",
    primary_metric="accuracy"
)

job.set_limits(
    timeout_minutes=60,
    trial_timeout_minutes=20,
    max_trials=5,
    enable_early_termination=True,
)


## Esto ya es un job invependiente

el mlflow.start_run() inicia la ejecución


In [ ]:
with mlflow.start_run():
    mlflow.log_params(params)
    
    mlflow.log_metric("accuracy", accuracy)
    mlflow.log_metric("precision", precision)
    mlflow.log_metric("recall", recall)
    mlflow.log_metric("f1", f1)
    
    mlflow.set_tag("Training info", "Iris dataset classification with Logistic Regression")
    
    # Predecir en el set de entreno
    # signatura necesita un ejemplo de entrada y salida para inferir el esquema
    # X_train → es el ejemplo de entrada (un array numérico con 4 columnas)
    # model.predict(X_train) → es el ejemplo de salida (las predicciones) -> vera que los posibles valores de salida son 0, 1 o 2 (las clases de iris)
    y_ejem = model.predict(X_train)
    X_ejem = X_train     
    signatures = infer_signature(X_ejem, y_ejem)

    # logging the model
    model_info = mlflow.sklearn.log_model(
        sk_model=model,
        artifact_path="iris_classifier",
        signature=signatures,
        input_example=X_train,
        registered_model_name="tracking-quickstart-iris-classifier"
    )
    
    # cargar el modelo
    loaded_model = mlflow.sklearn.load_model(model_uri=model_info.model_uri)
    
    predictions = loaded_model.predict(X_test)
    
    iris_feature_names = datasets.load_iris().feature_names
    
    result = pd.DataFrame(X_test, columns=iris_feature_names)
    result["actual_class"] = y_test
    result["predicted_class"] = predictions
    
    result[:4]